# Silver - ibge_renda_uf

Desenvolvido por: Ygor Moraes

Este notebook cria a Silver auxiliar `ibge_renda_uf`, usada para enriquecer clientes por perfil socioeconômico da UF.

Regras aplicadas:
- criar uma linha por UF brasileira;
- manter renda média per capita por UF;
- informar ano de referência e fonte do dado;
- gravar em Delta para uso nas Golds de clientes.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminho e parâmetros da Silver IBGE.

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    lit,
    current_timestamp,
    when
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

SILVER_IBGE_RENDA_UF_TABLE = "ibge_renda_uf"
SILVER_IBGE_RENDA_UF_PATH = f"{SILVER_BASE_PATH}{SILVER_IBGE_RENDA_UF_TABLE}"

SILVER_WRITE_MODE = "overwrite"

ANO_REFERENCIA = 2025
FONTE_IBGE = "IBGE - PNAD Contínua - rendimento domiciliar per capita por UF"

IBGE_REQUIRED_COLUMNS = [
    "uf",
    "nome_uf",
    "renda_media_per_capita",
    "ano_referencia",
    "fonte",
    "silver_processed_at"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Destino Silver IBGE: {SILVER_IBGE_RENDA_UF_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")
print(f"Ano de referência: {ANO_REFERENCIA}")

In [0]:
# Cria a base auxiliar de renda per capita por UF.

dados_ibge_renda_uf = [
    ("RO", "Rondônia", 1991.00),
    ("AC", "Acre", 1392.00),
    ("AM", "Amazonas", 1484.00),
    ("RR", "Roraima", 1878.00),
    ("PA", "Pará", 1420.00),
    ("AP", "Amapá", 1697.00),
    ("TO", "Tocantins", 2036.00),
    ("MA", "Maranhão", 1219.00),
    ("PI", "Piauí", 1546.00),
    ("CE", "Ceará", 1390.00),
    ("RN", "Rio Grande do Norte", 1819.00),
    ("PB", "Paraíba", 1543.00),
    ("PE", "Pernambuco", 1600.00),
    ("AL", "Alagoas", 1422.00),
    ("SE", "Sergipe", 1697.00),
    ("BA", "Bahia", 1465.00),
    ("MG", "Minas Gerais", 2353.00),
    ("ES", "Espírito Santo", 2249.00),
    ("RJ", "Rio de Janeiro", 2794.00),
    ("SP", "São Paulo", 2956.00),
    ("PR", "Paraná", 2762.00),
    ("SC", "Santa Catarina", 2809.00),
    ("RS", "Rio Grande do Sul", 2839.00),
    ("MS", "Mato Grosso do Sul", 2454.00),
    ("MT", "Mato Grosso", 2335.00),
    ("GO", "Goiás", 2407.00),
    ("DF", "Distrito Federal", 4538.00)
]

schema_ibge_renda_uf = StructType([
    StructField("uf", StringType(), False),
    StructField("nome_uf", StringType(), False),
    StructField("renda_media_per_capita", DoubleType(), False)
])

df_ibge_renda_uf = (
    spark
    .createDataFrame(dados_ibge_renda_uf, schema=schema_ibge_renda_uf)
    .withColumn("ano_referencia", lit(ANO_REFERENCIA).cast("int"))
    .withColumn("fonte", lit(FONTE_IBGE))
    .withColumn("silver_processed_at", current_timestamp())
)

total_linhas = df_ibge_renda_uf.count()

print("Base IBGE criada em memória.")
print(f"Total de linhas: {total_linhas}")

df_ibge_renda_uf.printSchema()

In [0]:
# Valida se a base possui 27 UFs, sem nulos e sem duplicidade.

colunas_ibge = df_ibge_renda_uf.columns

colunas_ausentes = [
    c for c in IBGE_REQUIRED_COLUMNS
    if c not in colunas_ibge
]

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes: {colunas_ausentes}")

df_validacao_ibge = df_ibge_renda_uf.select(
    count("*").alias("total_linhas"),
    countDistinct("uf").alias("total_ufs_distintas"),
    count(when(col("uf").isNull(), True)).alias("uf_nula"),
    count(when(col("nome_uf").isNull(), True)).alias("nome_uf_nulo"),
    count(when(col("renda_media_per_capita").isNull(), True)).alias("renda_media_nula"),
    count(when(col("ano_referencia").isNull(), True)).alias("ano_referencia_nulo"),
    count(when(col("fonte").isNull(), True)).alias("fonte_nula"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo")
)

display(df_validacao_ibge)

validacao_ibge = df_validacao_ibge.collect()[0]

duplicados_uf = (
    df_ibge_renda_uf
    .groupBy("uf")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total de linhas: {validacao_ibge['total_linhas']}")
print(f"Total de UFs distintas: {validacao_ibge['total_ufs_distintas']}")
print(f"UFs duplicadas: {duplicados_uf}")

if validacao_ibge["total_linhas"] != 27:
    raise Exception("Erro: a base IBGE deve conter exatamente 27 linhas.")

if validacao_ibge["total_ufs_distintas"] != 27:
    raise Exception("Erro: a base IBGE deve conter exatamente 27 UFs distintas.")

if duplicados_uf > 0:
    raise Exception("Erro: existem UFs duplicadas na base IBGE.")

if validacao_ibge["uf_nula"] > 0:
    raise Exception("Erro: existem registros com UF nula.")

if validacao_ibge["nome_uf_nulo"] > 0:
    raise Exception("Erro: existem registros com nome_uf nulo.")

if validacao_ibge["renda_media_nula"] > 0:
    raise Exception("Erro: existem registros com renda_media_per_capita nula.")

if validacao_ibge["ano_referencia_nulo"] > 0:
    raise Exception("Erro: existem registros com ano_referencia nulo.")

if validacao_ibge["fonte_nula"] > 0:
    raise Exception("Erro: existem registros sem fonte.")

if validacao_ibge["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

print("Validação OK: base IBGE em memória aprovada.")

In [0]:
# Regrava a Silver IBGE em Delta mantendo o schema usado pelas Golds.

(
    df_ibge_renda_uf
    .write
    .format("delta")
    .options(**adls_options)
    .mode(SILVER_WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(SILVER_IBGE_RENDA_UF_PATH)
)

print("Silver IBGE gravada com sucesso.")
print(f"Caminho: {SILVER_IBGE_RENDA_UF_PATH}")

In [0]:
# Lê a Silver gravada e valida volume, duplicidade e schema final.

df_ibge_renda_uf_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_IBGE_RENDA_UF_PATH)
)

total_linhas_saved = df_ibge_renda_uf_saved.count()

total_ufs_saved = (
    df_ibge_renda_uf_saved
    .select(countDistinct("uf"))
    .collect()[0][0]
)

duplicados_uf_saved = (
    df_ibge_renda_uf_saved
    .groupBy("uf")
    .count()
    .filter(col("count") > 1)
    .count()
)

colunas_ibge_saved = df_ibge_renda_uf_saved.columns

colunas_ausentes_saved = [
    c for c in IBGE_REQUIRED_COLUMNS
    if c not in colunas_ibge_saved
]

print(f"Total linhas em memória: {total_linhas}")
print(f"Total linhas gravadas: {total_linhas_saved}")
print(f"Total UFs gravadas: {total_ufs_saved}")
print(f"UFs duplicadas na Silver gravada: {duplicados_uf_saved}")

if total_linhas_saved != total_linhas:
    raise Exception("Erro: quantidade gravada diferente da quantidade em memória.")

if total_linhas_saved != 27:
    raise Exception("Erro: a Silver IBGE gravada deve conter exatamente 27 linhas.")

if total_ufs_saved != 27:
    raise Exception("Erro: a Silver IBGE gravada deve conter exatamente 27 UFs distintas.")

if duplicados_uf_saved > 0:
    raise Exception("Erro: existem UFs duplicadas na Silver IBGE gravada.")

if colunas_ausentes_saved:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver gravada: {colunas_ausentes_saved}")

df_ibge_renda_uf_saved.printSchema()

print("Validação final da Silver IBGE OK.")